# 🤖 Train SmolVLA for 15,000 Steps on Google Colab

This notebook guide helps you train **SmolVLA** (Hugging Face's lightweight Vision-Language-Action robotic policy) using Google Colab. 

### ⚙️ System Requirements
- Google Colab GPU runtime (T4, L4, or A100 are recommended).
- Local pre-converted dataset package (approx. 235 MB) or uploaded to the Hugging Face Hub.

## 🛠️ Step 1: System and GPU Verification
Verify that a GPU is available for training.

In [ ]:
!nvidia-smi

import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

## 📦 Step 2: Clone and Install LeRobot
We will clone the official LeRobot repository and install the package with support for the `smolvla` policy.

In [ ]:
# Clone the repository
!git clone https://github.com/huggingface/lerobot.git

# Navigate to the repository directory
%cd lerobot

# Install LeRobot with smolvla dependencies
!pip install -e ".[smolvla]"

# Install monitoring and extra utilities
!pip install wandb pandas jinja2

## 🔑 Step 3: Login to Hugging Face Hub (Optional)
Log in if you plan to fetch your dataset from the HF Hub or push your trained checkpoints back to the Hub.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 🔄 Step 4: Import Your Dataset
Choose one of the following two methods to bring your dataset into Google Colab.

### 🌟 Method A: Stream from Hugging Face Hub (Recommended)
If you uploaded your dataset to Hugging Face Hub using the instructions in `prepare_dataset_colab.sh`, set your repository ID below. The training script will download it automatically.

In [ ]:
# Set this to your Hugging Face dataset identifier
HF_REPO_ID = "your_hf_username/ur5_cube"
print(f"Method A Configured - Dataset Repo ID: {HF_REPO_ID}")

### 📁 Method B: Upload ZIP Archive (Google Drive)
If you zipped your dataset (`local/ur5_cube`) from your local machine (only ~235 MB) and uploaded it to your Google Drive, run the cell below to mount Google Drive and unzip the dataset directly into the Hugging Face cache.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Specify the path to your uploaded zip file in Google Drive
# (Default: root of MyDrive folder)
ZIP_PATH = "/content/drive/MyDrive/ur5_cube_converted.zip"

# 3. Extract to the Hugging Face cache directory
import os
cache_dir = os.path.expanduser("~/.cache/huggingface/lerobot")
os.makedirs(cache_dir, exist_ok=True)

!unzip -q {ZIP_PATH} -d {cache_dir}
print("✔ Dataset successfully extracted to LeRobot local cache!")

# Set the identifier to refer to your local cache dataset
HF_REPO_ID = "local/ur5_cube"
print(f"Method B Configured - Dataset Repo ID: {HF_REPO_ID}")

## 📊 Step 5: Login to Weights & Biases (WandB)
Logging into WandB allows you to view real-time learning curves, training loss, and learning rates on your dashboard.

In [ ]:
import wandb
wandb.login()

## 🚀 Step 6: Start SmolVLA Training
Launch SmolVLA training for **15,000 steps** (mirroring the settings in `train_smolvla_cube.sh`). Checkpoints will be saved every 1,000 steps.

This script automatically checks the GPU topology. If multiple GPUs are present, it launches using `torchrun` for distributed training, otherwise it uses standard single-device execution.

> 💡 *Tip: If you encounter Out of Memory (OOM) errors on a standard T4 GPU (16GB VRAM), adjust the batch size by lowering `--batch_size=16` to `--batch_size=8` or `--batch_size=4`.*

In [ ]:
import subprocess
import torch

# Detect number of GPUs
num_gpus = torch.cuda.device_count()
print(f"Detected {num_gpus} GPU(s)")

cmd = []
if num_gpus > 1:
    cmd += ["torchrun", f"--nproc_per_node={num_gpus}", "-m", "lerobot.scripts.lerobot_train"]
else:
    cmd += ["python", "-m", "lerobot.scripts.lerobot_train"]

cmd += [
    f"--dataset.repo_id={HF_REPO_ID}",
    "--policy.type=smolvla",
    "--policy.repo_id=local/smolvla-ur5-cube",
    "--policy.push_to_hub=false",
    "--policy.device=cuda",
    "--batch_size=16",
    "--steps=15000",
    "--save_freq=1000",
    "--eval.n_episodes=0",
    "--eval.batch_size=0",
    "--save_checkpoint=true",
    "--wandb.enable=true",
    "--policy.compile_model=false",
    "--output_dir=/content/lerobot_output_smolvla_cube"
]

print("🚀 Launching Training:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 💾 Step 7: Export and Back Up Checkpoints
When training is complete, pack the checkpoints into a zip archive and copy them to your Google Drive to prevent loss when the Colab instance shuts down.

In [ ]:
import os
import shutil

OUTPUT_DIR = "/content/lerobot_output_smolvla_cube"
ZIP_DEST = "/content/lerobot_output_smolvla_cube.zip"
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/"

if os.path.exists(OUTPUT_DIR):
    print("📦 Packaging checkpoints...")
    !zip -r {ZIP_DEST} {OUTPUT_DIR}
    
    # If Drive is mounted, copy it over
    if os.path.exists(DRIVE_BACKUP_DIR):
        print(f"💾 Copying checkpoints archive to Google Drive...")
        try:
            shutil.copy(ZIP_DEST, os.path.join(DRIVE_BACKUP_DIR, "lerobot_output_smolvla_cube.zip"))
            print("✅ Successfully backed up checkpoints archive to Google Drive!")
        except Exception as e:
            print(f"❌ Failed copying to Google Drive: {e}")
    else:
        print("ℹ Google Drive not mounted. You can download the zip file from the file manager on the left panel.")
else:
    print("❌ Checkpoint directory not found. Did the training finish successfully?")